# Legal Metrology ML: Explainability with SHAP

**Objective:** Interpret machine learning model behavior on weighing instrument verification records.

> **CRITICAL LEGAL NOTICE:**  
> SHAP (SHapley Additive exPlanations) values provide mathematical attribution of which feature values  
> influenced the machine learning model's output probability.  
> **They are NOT legal arguments or statutory grounds for rejection.**  
> Statutory rejection MUST always cite specific clauses in the applicable Legal Metrology rules (e.g. OIML R76 / General Rules MPE limits).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.predict import VerificationPredictor
from src.explainability.shap_explainer import MetrologyShapExplainer

## 1. Load Trained Predictor & SHAP Explainer

In [ ]:
predictor = VerificationPredictor(model_dir=project_root / "models")
print("Model Version:", predictor.model_version)
print("Loaded Explainer:", predictor.shap_explainer is not None)

## 2. Explain Sample Verification Cases

In [ ]:
import json
sample_file = project_root / "data" / "synthetic" / "sample_verification.json"
with open(sample_file, "r") as f:
    samples = json.load(f)["samples"]

for case in samples:
    print("\n" + "="*60)
    print("Case:", case["case_name"])
    res = predictor.predict_verification(case["data"])
    print("Rules Engine Result:", res["rules_engine_result"])
    print("ML Prediction:      ", res["prediction"], f"(Confidence: {res['confidence']:.2f}, Risk: {res['risk_score']:.2f})")
    print("Final Determination:", res["final_result"])
    print("Contributing Factors:")
    for expl in res.get("explanation", []):
        print(f"  - {expl['feature']}: {expl['impact'].upper()} ({expl['direction']})")